1. Датасет ml-latest.
2. Вспомнить подходы, которые мы разбирали.
3. Выбрать понравившийся подход к гибридным системам.
4. Написать свою.

In [3]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from surprise import Reader, Dataset, SVD
from surprise.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

In [4]:
df_movies = pd.read_csv('ml_latest_small/movies.csv')
df_ratings = pd.read_csv('ml_latest_small/ratings.csv')

In [5]:
print(df_movies.shape)
print(df_ratings.shape)

(9742, 3)
(100836, 4)


In [6]:
df_movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [7]:
df_ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


## Контентная фильтрация:

In [8]:
def create_movie_genres_matrix(df_movies, df_ratings):

    '''
    Создание матрицы для фильмов и жанров (OneHotEncoding)
    '''

    df_movies_tmp = df_movies.copy()

    df_movies_tmp.genres.str.split('|')

    all_genres = set()
    for genres in df_movies_tmp.genres:
        all_genres.update(genres)
    all_genres = sorted(list(all_genres))

    # movie_genres_matrix - матрица размером NxM,
    # где N - кол-во фильмов, M - кол-во жанров
    # На пересечении строки с фильмом и столбца с жанром стоит 1, если фильмо относится к этому жанру
    movie_genres_matrix = np.zeros(shape = (len(df_movies_tmp), len(all_genres)))
    for i, genres in enumerate(df_movies_tmp['genres']):
        for genre in genres:
            genre_idx = all_genres.index(genre)
            movie_genres_matrix[i, genre_idx] = 1

    movie_avg_ratings = df_ratings.groupby('movieId')['rating'].mean()

    return movie_genres_matrix, movie_avg_ratings

In [12]:
def get_content_recommendation(df_movies, df_ratings, user_id, n_recommendations = 10):
    '''
    Контентные рекомендации
    '''

    # Импортируем значения из метода создания матрицы для фильмов и жанров
    movie_genres_matrix, movie_avg_ratings = create_movie_genres_matrix(df_movies, df_ratings)

    # Фильмы с высокой оценкой от пользователя
    user_ratings = df_ratings[df_ratings['userId'] == user_id]
    linked_movies = user_ratings[user_ratings['rating'] >= 4.0]['movieId'].values

    # Если от пользователя нет хороших оценок, то выводим фильмы с самыми высокими средними оценками по всему набору данных
    if len(linked_movies) == 0:
        top_movies = movie_avg_ratings.sort_values(ascending = False).head(n_recommendations).index
        return pd.DataFrame({'movieId': top_movies, 'score': 1.0})


    linked_indices = df_movies[df_movies['movieId'].isin(linked_movies)].index

    # Создаем профиль клиента
    user_genre_profile = movie_genres_matrix[linked_indices].mean(axis = 0)

    # По всем фильмам создаем массив схожести с пользовательским профилем
    similarities = cosine_similarity([user_genre_profile], movie_genres_matrix)[0]
    df_similarity = pd.DataFrame({
        'movieId': df_movies['movieId'],
        'score': similarities
    })

    watched_movies = df_ratings[df_ratings['userId'] == user_id]['movieId'].values

    # Выводим рекомендации по всем фильмам, кроме просмотренных пользователем
    recommendations = df_similarity[~df_similarity['movieId'].isin(watched_movies)]
    recommendations = recommendations.sort_values('score', ascending = False).head(n_recommendations)

    return recommendations

Выведем результат для пользователя 1:

In [13]:
get_content_recommendation(df_movies, df_ratings, user_id = 1, n_recommendations = 5)

,movieId,score
5490,26340,0.945323
6448,51939,0.945323
7374,79139,0.945323
8357,108932,0.945323
488,558,0.945323


## Коллаборативная фильтрация:

In [14]:
def get_collaborative_recommendations(df_movies, df_ratings, user_id, n_recommendations = 10):
    '''
    Коллаборативные рекомендации
    '''

    reader = Reader(rating_scale = (0.5, 5))
    data = Dataset.load_from_df(df = df_ratings[['userId', 'movieId', 'rating']], reader = reader)

    trainset, testset = train_test_split(data = data, test_size = .2, random_state = 42)

    model = SVD(n_factors = 100, n_epochs = 30, lr_all = 0.005, reg_all = 0.02)
    model.fit(trainset)

    watched_movies = df_ratings[df_ratings['userId'] == user_id]['movieId'].values
    unwatched_movies = df_movies[~df_movies['movieId'].isin(watched_movies)]['movieId'].values

    predictions = []

    for movie_id in unwatched_movies:
        predict = model.predict(user_id, movie_id)
        predictions.append({
            'movieId': movie_id,
            'score': predict.est
        })

    recommendations = pd.DataFrame(predictions)
    recommendations = recommendations.sort_values('score', ascending = False).head(n_recommendations)

    return recommendations


Выведем результат для пользователя 1:

In [15]:
get_collaborative_recommendations(df_movies, df_ratings, user_id = 1, n_recommendations = 5)

,movieId,score
648,910,5.0
828,1193,5.0
7536,91529,5.0
775,1104,5.0
2377,3468,5.0


## Взвешенное объединение: